<a href="https://colab.research.google.com/github/YakubuNaat/Materials-Properties-Prediction-with-DFT-and-ML/blob/main/featurization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!apt install python3-mpi4py cython3 libxc-dev gpaw-data
!pip -q install gpaw jarvis-tools spglib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install ase
!pip install pymatgen
!pip install matminer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 26.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.0/809.0 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.1/739.1 kB 34.4 MB/s eta 0:00:00
  Created wheel for bibtexparser: filename=bibtexparser-1.4.3-py3-none-any.whl size=43549 sha256=e5341a13e399fc4d35dbec94e5b8ef307559bdff38b10c15b02208185dafe5e1
  Stored in directory: /root/.cache/pip/wheels/16/fb/76/306387739cf9d53b1c39b0c8aadb

In [ ]:
import os
from ase.io import write
from ase.build import bcc100

# List of transition metals (excluding La and Ac)
transition_metals = ['Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn',
                     'Y', 'Zr', 'Nb', 'Mo', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd',
                     'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg',
                     'Pb', 'Bi','Sn', 'Sb']

# Define the path to save the output files
output_dir = '/content/drive/MyDrive/DR.CAROLINE/Pristine'
os.makedirs(output_dir, exist_ok=True)

# Loop through each transition metal and create a modified CIF file
for metal in transition_metals:
    # Create the bcc (111) slab of iron
    original_atoms = bcc100(symbol= "Fe", size=(3, 3, 3), a=2.87, vacuum=10, orthogonal=False)

    # Make a copy of the original atoms
    atoms = original_atoms.copy()

    # Replace a specific atom (e.g., the 5th atom) with the transition metal
    atoms[22].symbol = metal

    # Write the modified structure to a new CIF file
    output_filename = os.path.join(output_dir, f'Fe_dopedwith_{metal}.cif')
    write(output_filename, atoms)

print("Fe atoms have been successfully been replaced with each transition metal and saved to separate CIF files.")

Fe atoms have been successfully been replaced with each transition metal and saved to separate CIF files.


In [ ]:
from ase.visualize import view
from ase.io import read, write
atoms = read('/content/cifs/Fe_dopedwith_Au.cif')
view(atoms,viewer='x3d')

In [ ]:
import os
import pandas as pd
import ase
from pymatgen.io.ase import AseAtomsAdaptor
from ase.io import read
from matminer.featurizers.composition import ElementProperty

In [ ]:
import os
import pandas as pd
from pymatgen.core.composition import Composition
from matminer.featurizers.composition import ElementProperty
from ase.io import read

# Directory containing CIF files
cif_directory = "/content/drive/MyDrive/ DR.CAROLINE/finalwork/DOPED CLUSTERS"

# Define features and statistics
features = ['Number', 'MendeleevNumber', 'AtomicWeight', 'MeltingT',
            'Column', 'Row', 'CovalentRadius', 'Electronegativity',
            'NsValence', 'NpValence', 'NdValence', 'NfValence', 'NValence',
            'NsUnfilled', 'NpUnfilled', 'NdUnfilled', 'NfUnfilled', 'NUnfilled',
            'GSvolume_pa', 'GSbandgap', 'GSmagmom', 'SpaceGroupNumber']
stats = ['mean']

# Initialize featurizer
featurizer = ElementProperty(data_source='magpie', features=features, stats=stats)

# List to store data
data = []

# Loop through CIF files
for cif_file in os.listdir(cif_directory):
    if cif_file.endswith(".cif"):
        cif_path = os.path.join(cif_directory, cif_file)
        try:
            # Read CIF with ASE
            atoms = read(cif_path)

            # Get formula and convert to pymatgen Composition
            formula = atoms.get_chemical_formula()
            composition = Composition(formula)

            # Featurize
            features_vector = featurizer.featurize(composition)

            # Append data
            data.append([cif_file, composition.reduced_formula] + features_vector)

        except Exception as e:
            print(f"Error processing {cif_file}: {e}")

# Create DataFrame and save
columns = ["CIF_File", "Composition"] + featurizer.feature_labels()
df = pd.DataFrame(data, columns=columns)
df.to_csv("doped_Felix_features.csv", index=False)

print("Featurization complete. CSV saved as 'doped_Felix_features.csv'.")


Featurization complete. CSV saved as 'doped_Felix_features.csv'.


/usr/local/lib/python3.11/dist-packages/matminer/utils/data.py:326: UserWarning: MagpieData(impute_nan=False):
In a future release, impute_nan will be set to True by default.
                    This means that features that are missing or are NaNs for elements
                    from the data source will be replaced by the average of that value
                    over the available elements.
                    This avoids NaNs after featurization that are often replaced by
                    dataset-dependent averages.
  warnings.warn(f"{self.__class__.__name__}(impute_nan=False):\n" + IMPUTE_NAN_WARNING)
